# ⚖️ 체중 · 칼로리 분석기 v0.3

매일 아침 공복 체중과 그날 먹은 칼로리를 **메모에 적어 붙여넣으면** → 체중 추세를 그리고 → **내 기록으로 유지 칼로리를 계산**하고 → 목표 속도(주당 kg)에 맞춘 **추천 섭취 칼로리**를 알려 줘요.

**사용법**
1. 아래 ①의 `WEIGHT_MEMO` 칸에 체중 메모를 붙여넣기
2. ②에서 목표 속도(`TARGET_KG_PER_WEEK`)를 확인
3. 메뉴 `런타임(Runtime) → 모두 실행(Run all)`

**메모 쓰는 법** — 한 줄 = 하루
```
체중                ← 제목 줄 (있어도 없어도 됨)
10/03 84.0 2200     ← 날짜(월/일) 체중(kg) 먹은 칼로리(kcal)
10/04 83.8 2150
10/06 83.9          ← 칼로리를 모르면 안 써도 됨
```
- 야간 당직 끝난 아침처럼 **안 잰 날은 줄을 안 쓰면** 돼요.
- `2200kcal`처럼 kcal를 붙여 써도 돼요.
- 위에서 아래로 = 예전 → 최근. 12월 다음에 1월이 나오면 해가 바뀐 걸로 쳐요.

**데이터 흐름**
```
메모(글자) → parse_weight_memo() → df(표: 한 줄 = 하루) → ③ 7일 평균 · 주별 요약 · 그래프 → ④ 유지 칼로리 → ⑤ 추천 섭취 칼로리
```

## ① 체중 메모 붙여넣기
`"""` 와 `"""` 사이에 붙여넣어요. 헬스 기록과 **다른 메모**예요 (날짜가 꼭 필요해서).

In [ ]:
WEIGHT_MEMO = """
체중
10/03 84.0
"""

## ② 설정
**여기만 고치면** 아래 결과가 바뀌어요.

| 설정 | 뜻 |
|---|---|
| `TARGET_KG_PER_WEEK` | 목표 속도 (kg/주). 감량 `-0.5`, 유지 `0`, 증량 `0.25` |
| `INITIAL_MAINTENANCE` | 기록이 부족할 때 쓰는 유지 칼로리. 지금까지 관찰: 하루 2200kcal → 주당 0.5kg 감량 → 2200 + 550 = **2750** |
| `WINDOW_DAYS` | 유지 칼로리를 계산할 때 볼 최근 기간(일) |
| `KCAL_PER_KG` | 체중 1kg ≈ 7700kcal (대략값) |

In [ ]:
import pandas as pd
import numpy as np

TARGET_KG_PER_WEEK = -0.5     # 목표 속도 (kg/주): 감량 -0.5 / 유지 0 / 증량 0.25
INITIAL_MAINTENANCE = 2750    # 기록이 부족할 때 쓰는 유지 칼로리 (kcal)
WINDOW_DAYS = 14              # 최근 며칠로 유지 칼로리를 계산할지
KCAL_PER_KG = 7700            # 체중 1kg ≈ 7700kcal
START_YEAR = 2026             # 메모 첫 줄의 연도

## ③ 메모 → 표 (`parse_weight_memo`)
- **들어오는 것**: `WEIGHT_MEMO` (여러 줄짜리 글자)
- **나가는 것**: `df` (표: 한 줄 = 하루. 날짜 · 체중 · 칼로리 · 7일 평균), `skipped` (이해 못 한 줄)
- **고칠 곳**: 메모 쓰는 법이 바뀌면 `parse_weight_memo`

**7일 평균**: 체중은 수분 때문에 매일 0.5~1kg씩 출렁여요. 그래서 하루 체중 대신 **그날까지 7일 동안의 평균**으로 흐름을 봐요.

In [ ]:
def parse_weight_memo(memo, start_year):
    rows = []                 # 결과: 하루 = dict 하나
    skipped = []              # 이해 못 한 줄
    year = start_year
    last_month = None

    for line in memo.splitlines():
        tokens = line.replace(",", " ").split()      # "10/03 84.0 2200" → ['10/03', '84.0', '2200']
        if not tokens or tokens[0] == "체중":         # 빈 줄, 제목 줄
            continue

        date_parts = tokens[0].split("/")            # '10/03' → ['10', '03']
        if (len(tokens) < 2 or len(date_parts) != 2
                or not all(p.isdigit() for p in date_parts)
                or not tokens[1].replace(".", "", 1).isdigit()):
            skipped.append(line)
            continue

        month, day = int(date_parts[0]), int(date_parts[1])
        if last_month is not None and month < last_month:   # 12월 → 1월이면 해가 바뀜
            year += 1
        last_month = month

        kcal = None                                  # 칼로리는 선택
        if len(tokens) >= 3:
            k = tokens[2].lower().replace("kcal", "")
            if k.isdigit():
                kcal = int(k)

        rows.append({"날짜": pd.Timestamp(year, month, day),
                     "체중": float(tokens[1]),
                     "칼로리": kcal})
    return rows, skipped


rows, skipped = parse_weight_memo(WEIGHT_MEMO, START_YEAR)
df = pd.DataFrame(rows).sort_values("날짜").reset_index(drop=True)
df["7일 평균"] = df.rolling("7D", on="날짜")["체중"].mean().round(1)   # 그날 포함 최근 7일 평균 (체중계처럼 소수 첫째 자리)

print(f"{len(df)}일을 읽었어요. ({df['날짜'].min():%m/%d} ~ {df['날짜'].max():%m/%d})")
for line in skipped:
    print("⚠️ 이해 못 한 줄 (건너뜀):", line.strip())
df.tail(10)

## ③-2 주별 요약 + 체중 그래프
- **나가는 것**: 한 줄 = 한 주 (월요일 시작). 평균 체중 · 평균 칼로리 · 지난주보다 몇 kg 변했는지
- 그래프: 점 = 그날 체중, 선 = 7일 평균. **선**을 보세요.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1단계: 주마다 묶기 (월요일 시작)
weekly = df.groupby(df["날짜"].dt.to_period("W-SUN")).agg(
    기록일수=("체중", "count"),
    평균체중=("체중", "mean"),
    평균칼로리=("칼로리", "mean"),
)
weekly["지난주 대비(kg)"] = weekly["평균체중"].diff()        # 이번 주 평균 - 지난주 평균
weekly.index = [f"{p.start_time:%m/%d}~" for p in weekly.index]
print(weekly.round(1))

# 2단계: 그래프 (글자는 영어 — Colab 한글 폰트 문제 피하기)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["날짜"], df["체중"], "o", color="#9a9a96", markersize=5, label="daily weight")
ax.plot(df["날짜"], df["7일 평균"], color="#2a78d6", linewidth=2, label="7-day average")
ax.set_ylabel("weight (kg)")
ax.grid(axis="y", color="#e5e5e3")
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))   # 날짜를 월/일로
plt.show()

## ④ 유지 칼로리 (내 기록으로 역산)
- **들어오는 것**: 최근 `WINDOW_DAYS`일의 체중 · 칼로리
- **나가는 것**: `maintenance` (유지 칼로리)
- **고칠 곳**: 기간은 ②의 `WINDOW_DAYS`, 기록이 부족할 때의 값은 ②의 `INITIAL_MAINTENANCE`

```
유지 칼로리 = 평균 섭취 칼로리 − (하루 체중 변화 kg × 7700)
```
- **하루 체중 변화**는 최근 기간의 체중 점들에 **직선을 그어서** 기울기로 구해요. 하루하루 출렁임에 덜 흔들려요.
- 예) 평균 2200kcal를 먹고 하루 −0.071kg (주당 −0.5kg)씩 빠지면 → 2200 − (−0.071 × 7700) ≈ **2750kcal**
- 최근 기간에 체중 **10일 이상**, 칼로리 **7일 이상**이 있어야 계산해요. 모자라면 ②의 초기값을 써요.

In [ ]:
end = df["날짜"].max()
recent = df[df["날짜"] > end - pd.Timedelta(days=WINDOW_DAYS)]     # 최근 WINDOW_DAYS일
kcal_days = recent["칼로리"].dropna()                               # 칼로리를 쓴 날만

if len(recent) >= 10 and len(kcal_days) >= 7:
    x = (recent["날짜"] - recent["날짜"].min()).dt.days               # 0, 1, 2 … 일째
    slope = np.polyfit(x, recent["체중"], 1)[0]                       # 직선의 기울기 = 하루 체중 변화(kg)
    maintenance = kcal_days.mean() - slope * KCAL_PER_KG
    print(f"최근 {WINDOW_DAYS}일: 체중 {len(recent)}일 · 칼로리 {len(kcal_days)}일 기록")
    print(f"평균 섭취 {kcal_days.mean():.0f}kcal, 체중 변화 주당 {slope * 7:+.2f}kg")
    print(f"→ 유지 칼로리 ≈ {maintenance:.0f}kcal (내 기록으로 계산)")
else:
    slope = None
    maintenance = INITIAL_MAINTENANCE
    print(f"최근 {WINDOW_DAYS}일 기록이 부족해요 (체중 {len(recent)}/10일, 칼로리 {len(kcal_days)}/7일)")
    print(f"→ 유지 칼로리 = {maintenance}kcal (② 초기값)")

## ⑤ 추천 섭취 칼로리
- **들어오는 것**: ④의 `maintenance` + ②의 `TARGET_KG_PER_WEEK`
- **나가는 것**: 하루 추천 섭취 칼로리 + 확인용 숫자들

```
추천 섭취 = 유지 칼로리 + (목표 kg/주 × 7700 ÷ 7)
```
주당 −0.5kg이면 하루 약 550kcal를 덜 먹어요.
속도가 체중의 **주당 1%**를 넘으면 경고해요 (근육을 지키려면 주당 체중의 0.5~1% 감량 권장 — Helms 외 2014).

In [ ]:
daily_change = TARGET_KG_PER_WEEK * KCAL_PER_KG / 7          # 하루에 더/덜 먹을 칼로리
target_kcal = round((maintenance + daily_change) / 10) * 10   # 10kcal 단위로 반올림
current = df["7일 평균"].iloc[-1]                             # 지금 체중 (7일 평균)
rate_pct = TARGET_KG_PER_WEEK / current * 100                 # 주당 체중의 몇 %

result = {
    "지금 체중 (7일 평균)": f"{current:.1f}kg",
    "유지 칼로리": f"{maintenance:.0f}kcal",
    "목표 속도": f"주당 {TARGET_KG_PER_WEEK:+g}kg (체중의 {rate_pct:+.2f}%)",
    "추천 섭취": f"{target_kcal}kcal",
    "유지 대비": f"{daily_change:+.0f}kcal ({(target_kcal / maintenance - 1) * 100:+.0f}%)",
}
if slope is not None:
    result["실제 속도 (최근)"] = f"주당 {slope * 7:+.2f}kg"
if abs(rate_pct) > 1:
    result["⚠️ 주의"] = "주당 체중의 1%보다 빠른 속도예요"

pd.Series(result).to_frame("값")